In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [17]:
#Create the data frames
parks = pd.read_csv('Data/parks.csv')
species = pd.read_csv('Data/species.csv', dtype={13: 'str'})

In [4]:
#removing empty column 'Unnamed: 13'
species.drop(columns = ['Unnamed: 13'], inplace = True)

In [18]:
species.head(5)
species.info(verbose = False)

<class 'pandas.DataFrame'>
RangeIndex: 119248 entries, 0 to 119247
Columns: 14 entries, Species ID to Unnamed: 13
dtypes: str(14)
memory usage: 27.8 MB


In [6]:
parks.head(5)
## parks.info(verbose = False)

,Park Code,Park Name,State,Acres,Latitude,Longitude
0,ACAD,Acadia National Park,ME,47390,44.35,-68.21
1,ARCH,Arches National Park,UT,76519,38.68,-109.57
2,BADL,Badlands National Park,SD,242756,43.75,-102.50
3,BIBE,Big Bend National Park,TX,801163,29.25,-103.25
4,BISC,Biscayne National Park,FL,172924,25.65,-80.08


In [8]:
#Show unique values for the Conservation Status column

print(species['Conservation Status'].unique())

<ArrowStringArray>
[                  nan,  'Species of Concern',          'Endangered',
         'In Recovery',          'Threatened',        'Under Review',
 'Proposed Threatened',             'Extinct', 'Proposed Endangered',
            'Resident',             'Breeder',           'Migratory']
Length: 12, dtype: str


In [10]:
#Join the tables to group species by park and then by State (State par not done yet)

result_sp = species.merge(parks, how = 'left', on = 'Park Name')
result_sp[25:30]
## result_sp.info(verbose = False)

,Species ID,Park Name,Category,Order,Family,Scientific Name,Common Names,Record Status,Occurrence,Nativeness,Abundance,Seasonality,Conservation Status,Park Code,State,Acres,Latitude,Longitude
25,ACAD-1025,Acadia National Park,Mammal,Chiroptera,Vespertilionidae,Myotis leibii,"Eastern Small-Footed Myotis, Small-Footed Myotis",Approved,Not Present (Historical Report),Native,NaN,NaN,Species of Concern,ACAD,ME,47390,44.35,-68.21
26,ACAD-1026,Acadia National Park,Mammal,Chiroptera,Vespertilionidae,Myotis lucifugus,"Little Brown Bat, Little Brown Myotis",Approved,Present,Native,Common,NaN,Species of Concern,ACAD,ME,47390,44.35,-68.21
27,ACAD-1027,Acadia National Park,Mammal,Chiroptera,Vespertilionidae,Pipistrellus subflavus,"Eastern Pipistrelle, Northern Georgian Bat",Approved,Not Present (False Report),Not Native,NaN,NaN,NaN,ACAD,ME,47390,44.35,-68.21
28,ACAD-1028,Acadia National Park,Mammal,Lagomorpha,Leporidae,Lepus americanus,"Snowshoe Hare, Varying Hare",Approved,Present,Native,Common,Breeder,NaN,ACAD,ME,47390,44.35,-68.21
29,ACAD-1029,Acadia National Park,Mammal,Rodentia,Castoridae,Castor canadensis,"American Beaver, Beaver, Canadian Beaver",Approved,Present,Not Native,Common,Breeder,NaN,ACAD,ME,47390,44.35,-68.21


In [11]:
# I want to see endangered species in Maine 

state_sp = result_sp[(result_sp['State'] == 'OH') & (result_sp['Conservation Status'] == "Endangered")]
state_sp
## state_sp.info(verbose = False)

,Species ID,Park Name,Category,Order,Family,Scientific Name,Common Names,Record Status,Occurrence,Nativeness,Abundance,Seasonality,Conservation Status,Park Code,State,Acres,Latitude,Longitude
22843,CUVA-1019,Cuyahoga Valley National Park,Mammal,Chiroptera,Vespertilionidae,Myotis sodalis,Indiana Bat,Approved,Present,Native,Rare,NaN,Endangered,CUVA,OH,32950,41.24,-81.55


In [12]:
#Filter only the endangered species in the DataFrame
state_sp = result_sp[result_sp['Conservation Status'] == "Endangered"]

#Group and count how many endangered species each state has (realise some Parks run across multiple states)
list_prep = state_sp.groupby(['State'])['Species ID'].count()
## state_sp.info(verbose = False)
list_prep

State
AK            31
AR             4
AZ            16
CA            67
CA, NV        24
CO            10
FL            33
HI            84
KY            13
ME             3
MI             2
MN             1
ND             3
NM             4
NV             2
OH             1
OR             4
SC             4
SD             5
TN, NC        13
TX            13
UT            22
VA             4
WA             6
WY             3
WY, MT, ID     2
Name: Species ID, dtype: int64

In [21]:
## Since there are some endangered species in shared parks (parks spanning multiple states), I will create a dictionary of endangered species 
## in states by adding the value from shared parks to all the states they belong to
dict_st = {}

for i in list_prep.index:
    if not ',' in i:
        dict_st[i] = list_prep[i]

for i in list_prep.index:
    if ',' in i:
        states = i.split(', ')
        for s in states:
            if s in list_prep.index:
                dict_st[s] += list_prep[i]
            else:
                dict_st[s] = list_prep[i]


In [20]:
#Transform dictionary in Series and sort it
series_st = pd.Series(dict_st, name = 'Rank')
series_st = series_st.sort_values(ascending = False)

series_st

CA    91
HI    84
FL    33
AK    31
NV    26
UT    22
AZ    16
TX    13
TN    13
KY    13
NC    13
CO    10
WA     6
SD     5
WY     5
NM     4
VA     4
OR     4
AR     4
SC     4
ND     3
ME     3
MI     2
ID     2
MT     2
MN     1
OH     1
Name: Rank, dtype: int64

In [19]:
#Save the list of states in a csv file (output)

rank_st = series_st.to_csv('Data/rank.csv')
rank_st = pd.read_csv('Data/rank.csv')
rank_st


,Unnamed: 0,Rank
0,CA,91
1,HI,84
2,FL,33
3,AK,31
4,NV,26
5,UT,22
6,AZ,16
7,TX,13
8,TN,13
9,KY,13
